# Portfolio Risk Analysis with Python 

## Project Objective

The goal of this project is to develop a quantitative framework to analyse the risk of a portfolio of financial assets.

More specifically, the project aims to:

- Retrieve historical market data for multiple assets
- Compute statistical properties of asset returns (volatility, correlations)
- Estimate portfolio volatility using the covariance matrix
- Simulate potential market scenarios using Monte Carlo methods
- Evaluate portfolio risk using metrics such as Value at Risk (VaR) and Expected Shortfall (ES)
- Perform backtesting to assess the accuracy of the risk model
- Analyse how individual assets contribute to the overall portfolio risk

## Project steps

1. Introduction
2. Market Data Collection
3. Returns Analysis
4. Asset Volatility
5. Correlation Structure
6. Portfolio Volatility Estimation
7. Monte Carlo Simulation
8. Value at Risk & Expected Shortfall
9. VaR Backtesting
10. Stress Scenario Analysis
11. Risk Contributions
12. Conclusion



### Introduction

Understanding the risk of an investment portfolio is a central problem in quantitative finance. While investors often focus on returns, analysing potential losses and the sources of risk is equally important.

This project explores the risk characteristics of an equity portfolio using statistical tools and simulation methods implemented in Python. 

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import yfinance as yf

### Market Data Collection

The first step of the analysis consists in retrieving historical market data for the assets included in the portfolio.

To do so, we use the Python library yfinance, which allows us to download historical financial data directly from Yahoo Finance. The selected assets include individual equities as well as market indices in order to capture different sources of risk and diversification.

The dataset contains daily adjusted closing prices between 2019 and 2025.

In [3]:
assets = ["DSY.PA", "SAN.PA", "MC.PA", "BY6.F", "PAASI.PA", "^GSPC", "^STOXX50E"]

prices = yf.download(
    assets,
    start="2019-01-01",  
    end="2025-01-01",
    auto_adjust=True      
)["Close"]

prices = prices.dropna()
prices.head()
price.to_csv("prices.csv")
prices = pd.read_csv("prices.csv", index_col=0, parse_dates=True)

[*********************100%***********************]  7 of 7 completed


NameError: name 'price' is not defined

The resulting dataset contains the daily adjusted closing prices for each asset in the portfolio. 

Missing observations are removed to ensure a consistent time series across all assets. The dataset is also saved locally as a CSV file to allow reproducibility and faster loading in subsequent analyses.

These price series will be used in the next step to compute asset returns and analyse their statistical properties.

### Returns Analysis

In order to analyse the statistical behaviour of asset prices, we first compute logarithmic returns from the price series.

Log returns are commonly used in quantitative finance because they present several useful properties. In particular, they allow returns to be aggregated over time and often provide a better approximation of continuously compounded returns.

The log return of an asset is defined as:

r = ln(P(t) / P(t-1))

where P(t) represents the asset price at time t.

Once computed, these returns allow us to study the dynamics and variability of asset price movements.

In [ ]:
returns = np.log(prices / prices.shift(1))
returns = returns.dropna()

returns.head()

In [ ]:
returns["MC.PA"].plot(figsize=(10,4))
plt.title("LVMH Returns Over Time")
plt.xlabel("Time")
plt.ylabel("Return")
plt.show()

The resulting dataset contains the daily log returns of each asset in the portfolio.

These returns provide a clearer view of price fluctuations and will be used throughout the analysis to compute key statistical measures such as volatility, correlations and portfolio risk metrics.

The figure above illustrates the time evolution of the daily returns for LVMH, highlighting the variability and occasional large fluctuations that are typical in financial markets.

### Asset Volatility

Volatility is one of the most widely used measures of financial risk. It reflects the variability of asset returns and provides an indication of how much an asset price fluctuates over time.

In this section, we analyse the statistical properties of asset returns and compute their daily volatility. We also visualise the distribution of returns for one of the assets in the portfolio to better understand the variability and dispersion of price changes.

These statistics provide a first overview of the risk characteristics of each asset before analysing the portfolio as a whole.

In [4]:
stats = returns.describe().T
stats["vol_daily"] = returns.std()
stats[["mean","vol_daily"]]
print(stats.head())

returns["MC.PA"].hist(bins=50)
plt.title("Distribution of LVMH Returns")
plt.xlabel("Returns")
plt.ylabel("Frequency")
plt.show()

NameError: name 'returns' is not defined

The table above summarises the main statistical properties of asset returns, including their mean and daily volatility.

The histogram illustrates the distribution of LVMH daily returns. Financial returns often exhibit a concentration around zero with occasional large fluctuations, reflecting periods of higher market volatility.

## Annualised Volatility

To make volatility easier to interpret and compare across assets, it is common practice in finance to express volatility on an annual basis.

Assuming approximately 252 trading days per year, annualised volatility can be computed as:

σ_annual = σ_daily × √252

This transformation allows us to compare the risk of different assets on a yearly scale.

In [5]:
#daily volatility
vol_daily = returns.std()

#annual volatility
vol_annual = returns.std() * np.sqrt(252)

#table with both
vol_table = pd.DataFrame({
    "daily volatility": vol_daily,
    "annual volatility": vol_annual
})

vol_table.sort_values("annual volatility", ascending=False)

vol_annual.plot(kind="bar", figsize=(8,4))
plt.title("Assets Annual volatility")
plt.ylabel("Volatility")
plt.show()

NameError: name 'returns' is not defined

The figure above compares the annualised volatility of the assets included in the portfolio.

Higher volatility indicates greater variability in returns and therefore higher risk. Some assets may exhibit significantly higher volatility than others, which can have a strong impact on the overall risk of the portfolio.

### Correlation Structure

In portfolio analysis, correlations between assets play a crucial role in determining the overall level of risk. 

While volatility measures the risk of each asset individually, correlations describe how asset returns move relative to one another. Assets that move together tend to increase portfolio risk, whereas assets with lower or negative correlations can provide diversification benefits.

To analyse these relationships, we compute the correlation matrix of asset returns and visualise it using a heatmap.

In [ ]:
corr_matrix = returns.corr()
corr_matrix

plt.figure(figsize=(10,7))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Matrix of Asset Returns")
plt.xlabel("Assets")
plt.ylabel("Assets")
plt.show()

The heatmap above illustrates the correlation structure between the assets in the portfolio.

Highly positive correlations indicate that assets tend to move in the same direction, while lower correlations suggest diversification potential. Understanding these relationships is essential when constructing a portfolio, as diversification can significantly reduce overall portfolio risk.

### Portfolio Volatility estimation

After analysing the statistical properties of individual assets, we now estimate the risk of the portfolio as a whole.

Portfolio risk depends not only on the volatility of each asset but also on the correlations between them. In portfolio theory, the overall volatility of a portfolio can be computed using the covariance matrix of asset returns and the portfolio weights.

The annualised portfolio volatility is calculated using the following formula:

σ_p = √(wᵀ Σ w)

where:

- w represents the vector of portfolio weights  
- Σ represents the covariance matrix of asset returns  
- σ_p represents the portfolio volatility

In [ ]:
weight = {
    "MC.PA": 0.250,
    "BY6.F": 0.082,
    "PAASI.PA": 0.044,
    "^GSPC": 0.321,
    "^STOXX50E": 0.095,
    "DSY.PA": 0.095,
    "SAN.PA": 0.113
}


w = pd.Series(weight).reindex(returns.columns)

# annualised covariance matrix
cov_ann = returns.cov() * 252

# portfolio volatility
port_vol = np.sqrt(w.values.T @ cov_ann.values @ w.values)

print("Annualised Portfolio Volatility :", port_vol)

The resulting value represents the annualised volatility of the portfolio, which is a standard measure of portfolio risk.

This metric reflects the combined effect of asset volatility and correlations within the portfolio. A portfolio containing highly volatile or strongly correlated assets will typically exhibit higher overall risk.

The estimated mean returns and covariance matrix will also be used in the next step to simulate possible future market scenarios using Monte Carlo methods.

### Monte Carlo Simulation

To estimate potential future losses of the portfolio, we simulate a large number of possible market scenarios using a Monte Carlo approach.

Monte Carlo simulation is widely used in quantitative finance to model the uncertainty of financial markets. Instead of relying only on historical observations, the method generates many hypothetical scenarios based on the statistical properties of asset returns.

In this project, asset returns are assumed to follow a multivariate normal distribution characterised by:

- the vector of mean returns
- the covariance matrix of asset returns

Using these parameters, we simulate a large number of potential market outcomes.

In [ ]:
# parameters for Monte Carlo simulation
mu = returns.mean().values
Sigma = returns.cov().values

# Monte carlo method with 20k simulation
n_sims = 20000
rng = np.random.default_rng(42)

sim_returns = rng.multivariate_normal(mean=mu, cov=Sigma, size=n_sims)


The simulation generates 20,000 possible market scenarios consistent with the historical statistical properties of asset returns.

Each simulated scenario represents one possible set of returns for all assets in the portfolio. These simulated returns will then be used to compute the corresponding portfolio returns and estimate the distribution of potential portfolio losses.

This approach allows us to analyse the range of possible outcomes and evaluate risk metrics such as Value at Risk (VaR) and Expected Shortfall.

### Value at Risk and Expected Shortfall

After generating simulated market scenarios, we can analyse the distribution of potential portfolio losses.

Two widely used risk measures in financial risk management are Value at Risk (VaR) and Expected Shortfall (ES).

Value at Risk represents the maximum expected loss over a given time horizon at a specified confidence level. For example, a 95% VaR indicates that, under normal market conditions, losses should not exceed this threshold in 95% of cases.

Expected Shortfall complements VaR by measuring the average loss in the worst scenarios, that is, when losses exceed the VaR threshold. This metric therefore provides additional information about the severity of extreme losses.

In [ ]:
# Simulated portfolio returns
port_sim = sim_returns @ w

# Simulated losses
losses = -port_sim

# Value at Risk (95%)
var_95 = np.quantile(losses, 0.95)

print(f"VaR 95% (1 day) = {var_95:.4f}")
print(f"≈ {var_95*100:.2f}% potential loss")

V = 10000
var_eur = V * var_95
print(f"In 95% of cases, the portfolio should not lose more than ≈ {var_eur:.0f} € in one day")

In [ ]:
# Excepted Shortfall
tail_losses = losses[losses > var_95]
es_95 = tail_losses.mean()

print(f"Expected Shortfall 95% (1 day) = {es_95:.4f}")
print(f"≈ {es_95*100:.2f}% average loss in worst scenarios")

es_eur = V * es_95
print(f"Average loss in extreme scenarios ≈ {es_eur:.0f} €")

# loss Distribution 
plt.figure(figsize=(9,4))
plt.hist(losses, bins=80)
plt.axvline(var_95, linewidth=2, label="VaR 95%")
plt.axvline(es_95, linewidth=2, linestyle="--", label="ES 95%")
plt.legend()
plt.title("Simulated Portfolio Loss Distribution")
plt.xlabel("Loss")
plt.ylabel("Frequency")
plt.show()

The histogram above represents the distribution of simulated portfolio losses generated by the Monte Carlo simulation.

The vertical line indicates the Value at Risk (VaR) at the 95% confidence level, which represents the loss threshold that should not be exceeded in most market conditions.

The dashed line corresponds to the Expected Shortfall, which measures the average loss in the worst 5% of scenarios. This metric provides a more conservative measure of extreme risk.

These indicators allow us to quantify the potential downside risk of the portfolio under simulated market conditions.

### VaR Backtesting

Once the Value at Risk has been estimated using simulated scenarios, it is important to evaluate whether this risk measure provides a realistic representation of portfolio risk.

Backtesting consists of comparing the predicted VaR with the actual historical portfolio losses. If the model is well calibrated, the number of times where the realised loss exceeds the VaR threshold should approximately correspond to the chosen confidence level.

For a 95% VaR, we would expect around 5% of observations to exceed the VaR threshold. These exceedances are commonly referred to as exceptions.

The results indicate how often the realised portfolio losses exceed the estimated VaR threshold.

In [ ]:
# historical portfolio returns
port_returns_hist = returns @ w

# historical losses
losses_hist = -port_returns_hist

# VaR exceptions
exceptions = losses_hist > var_95
n_exceptions = exceptions.sum()
n_days = len(losses_hist)

exception_rate = n_exceptions / n_days

print(f"Number of days analysed: {n_days}")
print(f"Number of VaR exceptions: {n_exceptions}")
print(f"Exception rate: {exception_rate:.2%}")

### Stress Scenario Analysis

In addition to statistical risk measures such as Value at Risk, it is common practice in risk management to analyse how a portfolio behaves under extreme but plausible market scenarios.

Stress testing provides a complementary perspective to statistical risk measures by explicitly modelling extreme market conditions. This approach allows us to identify potential vulnerabilities in the portfolio and understand how severe losses could become during market crises.

In this example, we simulate a market stress scenario where equity markets experience significant negative shocks. Different assets are assigned different stress levels to reflect their typical sensitivity to market downturns.

In [ ]:
stress = pd.Series(0.0, index=returns.columns)

stress["DSY.PA"] = -0.12
stress["SAN.PA"] = -0.12
stress["MC.PA"]  = -0.12
stress["^STOXX50E"] = -0.12

stress["^GSPC"] = -0.08
stress["PAASI.PA"] = -0.10
stress["BY6.F"] = -0.25

# portfolio return under stress
port_stress_return = float(stress @ w)    
port_stress_loss = -port_stress_return

print(f"Portfolio loss under stress ≈ {port_stress_loss*100:.2f}%")
print(f"Portfolio loss under stress ≈ {V*port_stress_loss:.0f} €")

### Risk Contributions

While portfolio volatility provides a global measure of risk, it is also important to understand how individual assets contribute to this overall risk.

Risk contribution analysis allows us to decompose the portfolio volatility into contributions from each asset. This helps identify which assets are the main drivers of portfolio risk.

In this framework, the contribution of each asset depends on two main factors:

- its weight in the portfolio  
- its covariance with the portfolio returns

Analysing these contributions helps investors better understand the sources of risk and identify potential concentration effects within the portfolio.

In [ ]:
# Risk contributions (volatility)
cov_ann = returns.cov() * 252

# portfolio volatility
port_vol = np.sqrt(w.T @ cov_ann.values @ w)

# marginal contribution to risk
marginal_contrib = cov_ann.values @ w / port_vol

# risk contribution by asset
risk_contrib = w * marginal_contrib

risk_contrib_df = pd.DataFrame({
    "Weights": w,
    "Risk Contribution": risk_contrib
}, index=returns.columns)

risk_contrib_df["Contribution %"] = risk_contrib_df["Risk Contribution"] / port_vol
print(risk_contrib_df.sort_values("Contribution %", ascending=False))

risk_contrib_df["Contribution %"].plot(kind="bar", figsize=(8,4))
plt.title("Contribution of Each Asset to Portfolio Risk")
plt.ylabel("Share of Total Risk")
plt.xlabel("Assets")
plt.show()


The results highlight how much each asset contributes to the overall portfolio volatility.

An asset with a relatively small weight can still contribute significantly to portfolio risk if it exhibits high volatility or strong correlation with other assets.

This analysis therefore helps identify the main drivers of portfolio risk and can provide useful insights for potential portfolio rebalancing strategies.

### Conclusion

This project explored the risk characteristics of an equity portfolio using statistical analysis and simulation techniques implemented in Python.

Starting from historical market data, we analysed asset returns, estimated their volatility and examined the correlation structure between assets. These elements allowed us to evaluate the risk of the portfolio as a whole through the covariance matrix and to estimate its annualised volatility.

Using a Monte Carlo simulation framework, we generated thousands of potential market scenarios in order to model the distribution of possible portfolio outcomes. This approach made it possible to estimate key risk metrics widely used in finance, such as Value at Risk (VaR) and Expected Shortfall (ES), which quantify potential losses under normal and extreme conditions.

We also performed backtesting to evaluate how well the VaR model reflects historical losses, and conducted stress scenario analysis to assess how the portfolio might behave under severe market shocks.

Finally, the risk contribution analysis highlighted how individual assets influence the overall risk of the portfolio, showing that portfolio risk depends not only on asset volatility but also on correlations and portfolio weights.

Overall, this project illustrates how statistical methods and simulation techniques can be used to better understand portfolio risk and support investment decision-making.

Future improvements could include the implementation of more advanced models, such as time-varying volatility models, alternative return distributions, or the development of an interactive portfolio risk analysis tool.